# 01. Exploring the Sphaera Harvest

This notebook analyzes bibliographic records harvested from the National Library of Israel (NLI) Alma catalogue via its public SRU endpoint. 

The harvest targets Hebrew translations of and commentaries on Johannes de Sacrobosco's *De sphaera*, including:
- *Mar'eh ha-Ofanim* (מראה האופנים) – Solomon ben Abraham Avigdor's translation
- *Sefer ha-Galgal* (ספר הגלגל)
- Moses Almosnino's commentary *Beit Elohim* (בית אלהים)
- Commentaries by Mattathias Delacrut (דלקרוט) and Perets ben Isaac Delesquer (ליסקר)

---

### Objectives
1. **Query Metrics:** Examine how many records were retrieved for each query term and what proportion are manuscripts.
2. **Manuscript Filter Evaluation:** Evaluate the accuracy of MARC Leader position 06 (`t` = manuscript language material) and inspect non-manuscript records to check for false negatives.
3. **MARC Field Inventory:** Survey all MARC fields/subfields present across manuscript records and measure their fill rates.
4. **Case Study (JTS Ms. 9121):** Inspect the complete field-by-field record for manuscript `990000846580205171` to identify the MARC tags for shelfmarks, colophons, script types, dates/places of writing, and holding libraries.

## 1. Load Harvested Data & Summary Statistics

In [1]:
from pathlib import Path
import pandas as pd

# Path resolution: works from code/notebooks/ or project root
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "metadata").exists())
CSV_DIR = PROJECT_ROOT / "data" / "metadata" / "raw"

# Load summary table, all deduplicated records, and manuscripts only
df_summary = pd.read_csv(CSV_DIR / "_SUMMARY.csv")
df_all = pd.read_csv(CSV_DIR / "ALL__deduped.csv", dtype=str)
df_ms = pd.read_csv(CSV_DIR / "MANUSCRIPTS_ONLY.csv", dtype=str)

print(f"Total unique records harvested: {len(df_all)}")
print(f"Total manuscript records:        {len(df_ms)}")
print(f"Non-manuscript records:          {len(df_all) - len(df_ms)}")

Total unique records harvested: 399
Total manuscript records:        235
Non-manuscript records:          164


### Search Term Harvest Summary Table
The table below shows the reported total in Alma, the number of records downloaded, and how many are catalogued manuscripts.

In [2]:
# Display summary table formatted with percentages
df_summary_display = df_summary[["term", "reported_total", "downloaded", "manuscripts"]].copy()
df_summary_display["ms_percentage"] = (df_summary_display["manuscripts"] / df_summary_display["downloaded"] * 100).round(1).astype(str) + "%"
df_summary_display

,term,reported_total,downloaded,manuscripts,ms_percentage
0,Sacrobosco,85,85,72,84.7%
1,סאקרו בוסקו,96,96,87,90.6%
2,מראה האופנים,116,116,105,90.5%
3,ספר הגלגל,148,148,74,50.0%
4,אשפירא,14,14,10,71.4%
5,אישפירא,6,6,6,100.0%
6,תבנית האישפירה,2,2,2,100.0%
7,אלמושנינו בית אלהים,23,23,23,100.0%
8,דלקרוט,80,80,39,48.8%
9,ליסקר,41,41,13,31.7%


### Multi-Term Overlap in Deduplicated Records
Because multiple queries can retrieve the same manuscript (e.g. *Sacrobosco* and *Mar'eh ha-Ofanim*), `sphaera_harvest.py` tracks all matching query terms in the `query_terms` column.

In [3]:
# Most frequent query combinations among manuscripts
df_ms["query_terms"].value_counts().head(10).to_frame(name="record_count")

,record_count
query_terms,
ספר הגלגל,43
מראה האופנים,40
Sacrobosco | סאקרו בוסקו | מראה האופנים,35
דלקרוט,29
Sacrobosco | סאקרו בוסקו | ספר הגלגל,20
אלמושנינו בית אלהים,10
סאקרו בוסקו | מראה האופנים | אלמושנינו בית אלהים,8
ליסקר,7
Sacrobosco | סאקרו בוסקו | מראה האופנים | דלקרוט,5


## 2. Evaluate the Manuscript Filter (Leader Position 06)

In the MARC21 bibliographic format, the **Leader** is a fixed 24-character string at the beginning of each record. Position 06 indicates the **Type of Record**:
- `t` = Manuscript language material
- `a` = Language material (printed books, articles)
- `d` = Manuscript music
- `f` = Manuscript cartographic material
- `p` = Mixed materials (often archival collections)
- `k` = Two-dimensional nonprojectable graphic (drawings, photographs)

The script classifies a record as a manuscript if Leader pos 06 is in `'tdf'`. Let's test whether this filter missed any manuscripts (false negatives).

In [4]:
# Extract Leader position 06
def extract_leader_pos06(ldr):
    if isinstance(ldr, str) and len(ldr) > 6:
        return ldr[6]
    return "missing"

df_all["leader_06"] = df_all["LDR"].apply(extract_leader_pos06)

# Cross-tabulate Leader position 06 against the manuscript classification
ct = pd.crosstab(
    df_all["leader_06"], 
    df_all["ms_is_manuscript"], 
    margins=True, 
    margins_name="Total"
)
ct

ms_is_manuscript,no,yes,Total
leader_06,,,
a,107,0,107
e,1,0,1
i,1,0,1
k,37,0,37
p,18,0,18
t,0,235,235
Total,164,235,399


### Inspecting Non-Manuscript Records for False Negatives

Let's search records marked `ms_is_manuscript == "no"` for typical manuscript keywords in Hebrew and English (such as "כתב יד", "כ"י", "manuscript", "קולופון", "העתק") to see if any actual manuscripts were missed.

In [5]:
# Filter non-manuscripts
df_non_ms = df_all[df_all["ms_is_manuscript"] == "no"].copy()

keywords = ['כתב יד', 'כ"י', 'manuscript', 'קולופון', 'אוטוגרף']

def contains_keywords(row):
    text_fields = [
        str(row.get("ms_title", "")),
        str(row.get("ms_notes", "")),
        str(row.get("ms_subjects", "")),
        str(row.get("500$a", ""))
    ]
    full_text = " ".join(text_fields)
    return any(kw in full_text for kw in keywords)

suspicious = df_non_ms[df_non_ms.apply(contains_keywords, axis=1)]
print(f"Non-manuscript records containing manuscript-related keywords: {len(suspicious)}")

# Display key metadata for these records
suspicious[["ms_mms_id", "leader_06", "ms_title", "ms_date_place", "ms_notes"]].head(5)

Non-manuscript records containing manuscript-related keywords: 3


,ms_mms_id,leader_06,ms_title,ms_date_place,ms_notes
1,997011536248105171,a,The diagram as paradigm : cross-cultural appro...,"Washington, DC : Dumbarton Oaks Research Libra...",Chiefly papers presented at a conference held ...
275,997012801940805171,a,[קריאת התורה].,1959.,עם ברכה כתובה ביד; עם גלגל טקסט מסתובב. || הער...
310,990020342990205171,a,נועם המדות : ... דברי מוסר /,"שאלוניקי [ועוד] : דפוס יצחק ג'אחנון, ה'ר'ח'ב'ת...","כולל ליקוטים במוסר ובאמונה, בפילוסופיה ובחכמות..."


### Analysis of the Filter
- **Leader pos 06 = `t`** captures all 235 primary manuscript records.
- Records marked `"no"` with `leader_06 == 'a'` that mention manuscript keywords turn out to be:
  1. Printed editions referencing manuscript variants or quoting manuscript colophons (e.g. *No'am ha-Middot*, Salonica 1854).
  2. Scholarly articles or modern studies analyzing manuscripts (e.g. *The diagram as paradigm : cross-cultural approaches*).
  3. Printed ephemera with handwritten inscriptions (e.g. a Rosh Hashanah greeting card with a rotating paper wheel).
- **Conclusion:** The Leader pos 06 filter (`t`) is reliable for the NLI Alma/Ktiv catalogue.

## 3. Inventory of MARC Fields in Manuscript Records

Let's survey all MARC data fields and subfields present across the 235 manuscript records, calculating how many records contain each field.

In [6]:
# Select MARC field/subfield columns (e.g. '100$a', '245$a', etc.)
marc_cols = [c for c in df_ms.columns if "$" in c or c.isdigit() or c == "LDR"]

# Count non-null and non-empty entries per column
fill_counts = {}
for col in marc_cols:
    count = df_ms[col].dropna().apply(lambda v: str(v).strip() != "").sum()
    if count > 0:
        fill_counts[col] = count

df_field_inventory = pd.DataFrame(
    list(fill_counts.items()), 
    columns=["marc_field", "record_count"]
)
df_field_inventory["fill_rate"] = (df_field_inventory["record_count"] / len(df_ms) * 100).round(1).astype(str) + "%"
df_field_inventory = df_field_inventory.sort_values(by="record_count", ascending=False).reset_index(drop=True)

print(f"Total distinct MARC field/subfield tags used: {len(df_field_inventory)}")
print("\nTop 25 most frequent MARC fields across manuscript records:")
df_field_inventory.head(25)

Total distinct MARC field/subfield tags used: 179

Top 25 most frequent MARC fields across manuscript records:


,marc_field,record_count,fill_rate
0,001,235,100.0%
1,500$a,235,100.0%
2,710$9,235,100.0%
3,710$a,235,100.0%
4,710$c,235,100.0%
5,710$e,235,100.0%
6,710$x,235,100.0%
7,005,235,100.0%
8,903$a,235,100.0%
9,903$b,235,100.0%


## 4. Case Study: Complete Field-by-Field Inspection of JTS Ms. 9121

System number **`990000846580205171`** corresponds to the Hebrew text of *Mar'eh ha-Ofanim* (ff. 8a–14b) in the composite codex **JTS Ms. 9121** (The Jewish Theological Seminary of America, New York).

Let's inspect every populated field for this record to identify where key codicological and bibliographic data reside.

In [7]:
target_mms = "990000846580205171"
rec_series = df_ms[df_ms["ms_mms_id"] == target_mms]

if len(rec_series) == 0:
    # If not in ms table, look in df_all
    rec_series = df_all[df_all["ms_mms_id"] == target_mms]

rec = rec_series.iloc[0]

# Extract all populated fields
populated_fields = []
for col in df_all.columns:
    val = rec.get(col)
    if pd.notna(val) and str(val).strip() != "":
        populated_fields.append((col, str(val).strip()))

df_rec = pd.DataFrame(populated_fields, columns=["MARC_Tag_or_Field", "Value"])
print(f"Record {target_mms}: {len(df_rec)} populated fields.\n")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
df_rec

Record 990000846580205171: 98 populated fields.



,MARC_Tag_or_Field,Value
0,ms_added_names,שלמה בן אברהם אביגדור מתרגם
1,ms_author,יוהאנס דה סאקרו בוסקו
2,ms_date_place,"ש""ז (1547)."
3,ms_extent,"דף 8א-14ב (ספירת דפים מקורית: קיא ע""א-קיח ע""ב)."
4,ms_iiif_manifest,https://iiif.nli.org.il/IIIFv21/990000846580205171/manifest
5,ms_is_manuscript,yes
6,ms_ktiv_link,https://www.nli.org.il/he/discover/manuscripts/hebrew-manuscripts/viewerpage?vid=MANUSCRIPTS&docid=PNX_MANUSCRIPTS990000846580205171
7,ms_mms_id,990000846580205171
8,ms_notes,"בכה""י: ""מראה האופנים"" ""נקרא בלשונם טראקטאט מהאישפיראה"" || עם הקדמת המתרגם: ""שלמה בן אברהם אביגדור"". || עם תרשימים ועם ציור ספינה. || ראה: שטיינשניידר, HU, עמ' 643. || נושא נוסף: מדעים. אסטרונומיה || נושא נוסף: ציורים. ספינה || נושא נוסף: תרשימים"
9,ms_subjects,"Astronomy -- Early works to 1800 | Charts, diagrams, etc."


### Mapping Bibliographic Information to MARC Tags

From the record inspection above, we can identify the specific MARC tags used by NLI/Ktiv:

| Information | MARC Tag & Subfield | Content in JTS Ms. 9121 (`990000846580205171`) |
| :--- | :--- | :--- |
| **Shelfmark (Local / Holding)** | **`942$z`** & **`907$e`** | `Ms. 9121` (also `Acc. 4212`) in `942$z`; `NY JTS 9121` in `907$e` |
| **Microfilm Shelfmark (NLI)** | **`090$a`** & **`AVA$d`** | `F 49568` |
| **Colophon** | **`957$a`** | Full Hebrew text transcribed: *"נשלם זה הספר ... שנת ש"ז לפ"ק פה ק"ק פרנקבורט ... השפל יודא בר אורי הלוי ... ליווא אשכנז"* |
| **Script Type (כתב)** | **`958$a`** | `אשכנזית` (Ashkenazi script) |
| **Place & Date of Writing** | **`260$c`**, **`751$a`** | `260$c`: `ש"ז (1547)`; `751$a`: `Frankfurt (Oder) (Germany)` |
| **Holding Library** | **`710$a`**, **`942$a`**, **`597$a`** | `The Jewish Theological Seminary of America` (current owner) |
| **Parent Manuscript Linking** | **`773$w`** & **`773$t`** | `990000846560205171` (Host codex: *Kovetz targumei Shelomoh ben Avraham Avigdor*) |